# MES – Hypothesis 3: Time‑of‑Day Short (14:00 ET)

**Idea**: Sell short at 14:00 ET every day, exit after N minutes.  

**Rationale**: Intraday seasonality showed negative returns around 14:00.  

**Constraint**: Only trade within regular hours (8:30‑16:00 ET) – 14:00 is inside.

## 1. Setup and load IS data (Feb–Apr 2026, 1‑min bars)

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import warnings
warnings.filterwarnings('ignore')
import os


IS_PATH = "../../../../Local-Data/processed/MES_IS_1min.csv"
df_is = pd.read_csv(IS_PATH, parse_dates=['DateTime'], index_col='DateTime')
print(f"IS data: {len(df_is):,} rows")
df_is.head()

IS data: 87,799 rows


,Open,High,Low,Close,Volume
DateTime,,,,,
2026-02-01 13:52:00,6961.25,6961.25,6961.25,6961.25,1
2026-02-01 20:01:00,6939.50,6945.25,6928.25,6936.75,2717
2026-02-01 20:02:00,6937.25,6937.75,6925.75,6926.75,1939
2026-02-01 20:03:00,6926.50,6927.75,6920.00,6927.00,2748
2026-02-01 20:04:00,6927.00,6931.50,6926.00,6928.75,1374


## 2. Restrict to regular trading hours (8:30‑16:00 ET)

In [8]:
start_time = pd.Timestamp('2026-02-01 08:30:00').time()
end_time   = pd.Timestamp('2026-02-01 16:00:00').time()
df_is_reg = df_is.between_time(start_time, end_time).copy()
print(f"IS rows within regular hours: {len(df_is_reg):,}")


IS rows within regular hours: 28,460


## 3. Backtest function for fixed time entry

In [3]:
def backtest_time_short(df, entry_hour=14, entry_minute=0, hold_minutes=15):
    """
    Enter short at fixed time (entry_hour:entry_minute) every day.
    Exit after `hold_minutes` minutes. Returns trades and metrics.
    """
    df = df.copy()
    # Filter days where we have at least entry+hold minutes
    trades = []
    # Group by date
    for date, day_df in df.groupby(df.index.date):
        # Find the bar at exact entry time
        target_time = pd.Timestamp(date).replace(hour=entry_hour, minute=entry_minute)
        if target_time not in day_df.index:
            continue
        entry_idx = day_df.index.get_loc(target_time)
        if entry_idx + hold_minutes >= len(day_df):
            continue
        exit_dt = day_df.index[entry_idx + hold_minutes]
        entry_price = day_df.loc[target_time, 'Open']
        exit_price = day_df.loc[exit_dt, 'Close']
        # Short trade: entry - exit (positive if price falls)
        pnl_points = entry_price - exit_price
        trades.append(pnl_points * 5.0)   # USD
    return trades

## 4. Grid search over hold minutes (on IS)

In [9]:
hold_minutes_list = [5, 10, 15, 20, 25, 30]
results = []
for hold in hold_minutes_list:
    trades = backtest_time_short(df_is_reg, 14, 0, hold)
    if trades:
        total_pnl = sum(trades)
        wins = sum(1 for t in trades if t > 0)
        win_rate = wins / len(trades)
        gross_profit = sum(t for t in trades if t > 0)
        gross_loss = -sum(t for t in trades if t < 0)
        pf = gross_profit / gross_loss if gross_loss != 0 else 0
        results.append({'hold_min': hold, 'trades': len(trades), 'pnl_usd': total_pnl,
                        'win_rate': win_rate, 'profit_factor': pf})
results_df = pd.DataFrame(results).sort_values('profit_factor', ascending=False)
print("Results on IS (Feb–Apr, regular hours):")
print(results_df.to_string(index=False))

Results on IS (Feb–Apr, regular hours):
 hold_min  trades  pnl_usd  win_rate  profit_factor
       20      63  1021.25  0.571429       2.196193
       15      63   780.00  0.539683       2.142857
       30      63  1043.75  0.492063       2.074646
       25      63   798.75  0.507937       1.782130
       10      63   376.25  0.460317       1.570076
        5      63    26.25  0.523810       1.038391


## 5. Best hold minutes

In [10]:
best_hold = int(results_df.iloc[0]['hold_min']) if not results_df.empty else 15
print(f"Best hold minutes: {best_hold}")


Best hold minutes: 20


## 6. Validate on OOS (May 2026) using aggregated ticks

In [11]:
OOS_TICKS_PATH = "../../../../Local-Data/processed/MES_OOS_ticks.csv"
if not os.path.exists(OOS_TICKS_PATH):
    print("OOS ticks file not found. Cannot validate.")
else:
    df_ticks = pd.read_csv(OOS_TICKS_PATH, parse_dates=['DateTime'], index_col='DateTime')
    # Aggregate to 1-min OHLCV
    ohlc = df_ticks['Price'].resample('1min').ohlc()
    vol = df_ticks['Volume'].resample('1min').sum()
    ohlc.columns = ['Open', 'High', 'Low', 'Close']
    df_oos = ohlc.join(vol).rename(columns={'Volume': 'Volume'}).dropna()
    # Restrict to regular hours
    df_oos_reg = df_oos.between_time(start_time, end_time).copy()
    print(f"OOS regular hours rows: {len(df_oos_reg):,}")
    
    # Backtest with best hold
    trades_oos = backtest_time_short(df_oos_reg, 14, 0, best_hold)
    if trades_oos:
        total_pnl = sum(trades_oos)
        wins = sum(1 for t in trades_oos if t > 0)
        win_rate = wins / len(trades_oos)
        gross_profit = sum(t for t in trades_oos if t > 0)
        gross_loss = -sum(t for t in trades_oos if t < 0)
        pf = gross_profit / gross_loss if gross_loss != 0 else 0
        print("\n=== OOS VALIDATION (May 2026, regular hours) ===")
        print(f"Hold minutes: {best_hold}")
        print(f"Trades: {len(trades_oos)}")
        print(f"Win rate: {win_rate:.2%}")
        print(f"Profit factor: {pf:.2f}")
        print(f"Total net profit (USD): ${total_pnl:.2f}")
        if pf > 1.2:
            print("✅ Strategy passes validation.")
        else:
            print("❌ Strategy fails validation (PF <= 1.2).")
    else:
        print("No trades generated on OOS.")


OOS regular hours rows: 7,667

=== OOS VALIDATION (May 2026, regular hours) ===
Hold minutes: 20
Trades: 17
Win rate: 35.29%
Profit factor: 0.72
Total net profit (USD): $-157.50
❌ Strategy fails validation (PF <= 1.2).


In [13]:
if 'trades_oos' in locals() and trades_oos and pf > 1.2:
    np.random.seed(42)
    sim_pnls = []
    for _ in range(1000):
        sim = np.random.choice(trades_oos, size=len(trades_oos), replace=True)
        sim_pnls.append(sim.sum())
    sim_pnls = np.array(sim_pnls)
    prob_loss = (sim_pnls < 0).mean()
    print("\n=== MONTE CARLO (1000 sims, daily PnL) ===")
    print(f"Mean daily PnL: ${sim_pnls.mean():.2f}")
    print(f"Probability of loss: {prob_loss:.2%}")
    print(f"95% CI: [${np.percentile(sim_pnls,2.5):.2f}, ${np.percentile(sim_pnls,97.5):.2f}]")
    # Plot
    plt.hist(sim_pnls, bins=30, alpha=0.7, edgecolor='black')
    plt.axvline(x=0, color='red', linestyle='--')
    plt.title('Monte Carlo – MES Time‑of‑Day Short')
    plt.show()

## ❌ Hypothesis 3 – Time‑of‑Day Short (14:00 ET) – DISCARDED

### Parameters tested
- **Entry**: 14:00 ET every day  
- **Hold minutes**: 20 (best on IS)  
- **Trading hours**: regular session only (8:30‑16:00 ET)

### Performance
| Period | Trades | Win rate | Profit factor | Net profit (USD) |
|--------|--------|----------|---------------|------------------|
| In‑Sample (Feb–Apr) | 63 | 57.1% | 2.20 | +$1,021 |
| Out‑of‑Sample (May) | 17 | 35.3% | 0.72 | –$158 |

### Why discarded
- **No robustness**: excellent IS results but failed OOS (PF < 1.2).  
- **Overfitting risk**: the pattern did not hold in a different month.  
- **Conclusion**: not suitable for the competition.

### Next step for MES
- The only viable strategy for MES is **Hypothesis 2 (negative gap reversal)** – low frequency but profitable (PF 1.57 on OOS).  
- We will deploy it in NinjaTrader and move to more volatile instruments (MNQ, 6E, MCL, MGC) to increase trade frequency.